[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C18_Computer_Vision_Course/04_segmentation/04_segmentation.ipynb)

# 04 · 图像分割（纯 numpy 从零）

目标：把分割的核心指标与最简管线从零写出来——**逐类 IoU / Dice**、**mIoU（含空类边界）**、**Otsu 阈值**、**连通域标记（语义→实例）**、**形态学清理**、**软 Dice loss（类不平衡）**，每步 `assert` 验证。

**路线**：
1. IoU 与 Dice（同 TP/FP/FN，Dice=2·IoU/(1+IoU)）
2. mIoU（对类别 macro 平均，空类边界处理避免 NaN）
3. Otsu 自动阈值（最大类间方差）
4. 连通域标记（4/8 邻接，把语义掩码切成实例）
5. 形态学清理 + 软 Dice loss
6. ✏️ 练习 → 📖 答案 → 🧪 真实 optdigits 分割胶囊

> **本课纪律**：空类导致 0/0 必须显式处理（跳过或置 1），绝不让一个 NaN 污染整个 mIoU。

## 1 · IoU 与 Dice（像素集合）

对二值掩码：IoU=TP/(TP+FP+FN)，Dice=2TP/(2TP+FP+FN)。两者关系 **Dice=2·IoU/(1+IoU)**。
用预测/真值掩码验证这个恒等式。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def iou_mask(pred, gt):
    pred = pred.astype(bool); gt = gt.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    return inter / union if union > 0 else 1.0      # 都空 -> 约定 1

def dice_mask(pred, gt):
    pred = pred.astype(bool); gt = gt.astype(bool)
    inter = (pred & gt).sum()
    denom = pred.sum() + gt.sum()
    return 2 * inter / denom if denom > 0 else 1.0   # 都空 -> 约定 1

gt = np.zeros((8, 8), dtype=bool); gt[2:6, 2:6] = True       # 4x4 前景
pred = np.zeros((8, 8), dtype=bool); pred[3:7, 3:7] = True   # 偏移的 4x4
i = iou_mask(pred, gt); d = dice_mask(pred, gt)
print(f'IoU={i:.4f}  Dice={d:.4f}')
# 交=[3,6)x[3,6)=9; 并=16+16-9=23 -> IoU=9/23; Dice=2*9/32
assert abs(i - 9/23) < 1e-9 and abs(d - 18/32) < 1e-9
# 恒等式 Dice = 2*IoU/(1+IoU)
assert abs(d - 2*i/(1+i)) < 1e-9, 'Dice 应 = 2·IoU/(1+IoU)'
assert iou_mask(gt, gt) == 1.0 and dice_mask(gt, gt) == 1.0
print('✅ IoU/Dice 正确，且满足 Dice=2·IoU/(1+IoU)')

## 2 · mIoU：对类别平均（空类边界）

多类语义分割：对每个类别算 IoU（该类的像素集合），再**对类别**取平均（macro）。
**关键陷阱**：某类在 pred 和 gt 里都不存在 → 0/0=NaN。必须跳过或约定为 1，否则一个 NaN 毁掉整个 mIoU。

In [ ]:
def per_class_iou(pred, gt, num_classes):
    '''返回每类 IoU 列表; 该类在 pred、gt 中都不存在则记 NaN(稍后跳过)。'''
    ious = []
    for c_ in range(num_classes):
        p = (pred == c_); g = (gt == c_)
        inter = (p & g).sum(); union = (p | g).sum()
        ious.append(inter/union if union > 0 else np.nan)   # 空类 -> NaN
    return np.array(ious)

def mean_iou(pred, gt, num_classes):
    ious = per_class_iou(pred, gt, num_classes)
    valid = ious[~np.isnan(ious)]                    # 跳过空类(避免 NaN 污染)
    return float(valid.mean()) if len(valid) else 1.0

# 3 类标签图; 类 2 在两图中都不出现(空类)
gt = np.array([[0,0,1],[0,1,1],[0,0,1]])
pred = np.array([[0,0,1],[0,0,1],[0,0,1]])
ious = per_class_iou(pred, gt, 3)
print('per-class IoU:', np.round(ious, 3), '(类2为 NaN=空类)')
assert np.isnan(ious[2]), '空类应为 NaN'
m = mean_iou(pred, gt, 3)
print('mIoU (跳过空类) =', round(m, 4))
assert not np.isnan(m), 'mIoU 不应是 NaN(空类已跳过)'
assert 0 <= m <= 1
print('✅ mIoU 正确：对类别平均、空类被跳过(无 NaN 污染)')

## 3 · Otsu 自动阈值

遍历所有阈值，选使**前景/背景类间方差最大**的那个（等价类内方差最小，把两类分得最开）。
用双峰直方图的图验证：Otsu 阈值应落在两峰之间的谷底附近。

In [ ]:
def otsu_threshold(img, nbins=64):
    '''返回使类间方差最大的阈值(灰度值)。'''
    hist, edges = np.histogram(img.ravel(), bins=nbins, range=(0, 1))
    centers = (edges[:-1] + edges[1:]) / 2
    total = hist.sum(); p = hist / total
    best_t, best_var = centers[0], -1.0
    for k in range(1, nbins):
        w0 = p[:k].sum(); w1 = p[k:].sum()
        if w0 == 0 or w1 == 0:
            continue
        mu0 = (centers[:k] * p[:k]).sum() / w0
        mu1 = (centers[k:] * p[k:]).sum() / w1
        between = w0 * w1 * (mu0 - mu1) ** 2          # 类间方差
        if between > best_var:
            best_var, best_t = between, centers[k]
    return best_t

# 双峰: 一半像素~0.2, 一半~0.8
img = np.concatenate([rng.normal(0.2, 0.05, 500), rng.normal(0.8, 0.05, 500)])
img = np.clip(img, 0, 1).reshape(40, 25)
t = otsu_threshold(img)
print('Otsu 阈值 =', round(t, 3), '(应在两峰 0.2 和 0.8 之间)')
assert 0.3 < t < 0.7, 'Otsu 阈值应落在双峰之间的谷底'
fg = (img > t)
assert 0.3 < fg.mean() < 0.7, '阈值应把双峰大致对半分'
print('✅ Otsu 阈值落在双峰谷底，分割合理')

## 4 · 连通域标记：语义掩码 → 实例

二值前景里相邻像素聚成一个实例。用泛洪填充(BFS)给每个连通块一个编号。
4 邻接(上下左右) vs 8 邻接(含对角)会给出不同实例数。

In [ ]:
def connected_components(bw, connectivity=4):
    '''返回 (label_map, n_components)。0=背景, 1..n=各实例。'''
    bw = bw.astype(bool); H, W = bw.shape
    labels = np.zeros((H, W), dtype=int)
    if connectivity == 4:
        nbrs = [(-1,0),(1,0),(0,-1),(0,1)]
    else:
        nbrs = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
    cur = 0
    for i in range(H):
        for j in range(W):
            if bw[i, j] and labels[i, j] == 0:
                cur += 1
                stack = [(i, j)]; labels[i, j] = cur   # 泛洪
                while stack:
                    y, x = stack.pop()
                    for dy, dx in nbrs:
                        ny, nx = y+dy, x+dx
                        if 0<=ny<H and 0<=nx<W and bw[ny,nx] and labels[ny,nx]==0:
                            labels[ny, nx] = cur; stack.append((ny, nx))
    return labels, cur

# 两个分离的块 + 对角相连的情形
bw = np.zeros((6, 6), dtype=bool)
bw[0:2, 0:2] = True       # 块 1
bw[4:6, 4:6] = True       # 块 2(分离)
_, n4 = connected_components(bw, 4)
print('两个分离块 -> 4 邻接实例数:', n4)
assert n4 == 2, '两个分离块应是 2 个连通域'
# 对角相连: 4 邻接算 2 个, 8 邻接算 1 个
diag = np.zeros((4, 4), dtype=bool); diag[1,1] = True; diag[2,2] = True
_, nd4 = connected_components(diag, 4)
_, nd8 = connected_components(diag, 8)
print(f'对角相连: 4邻接={nd4}个, 8邻接={nd8}个')
assert nd4 == 2 and nd8 == 1, '对角像素: 4邻接分开、8邻接相连'
print('✅ 连通域标记正确，4/8 邻接差异符合预期')

## 5 · 形态学清理 + 软 Dice loss

连通域前先用开运算去散点（避免噪点被当成实例）。
软 Dice loss = 1 − 软Dice（用概率而非硬阈值，可微，加 ε 防除零），对类不平衡鲁棒。

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
def erode(bw):
    p = np.pad(bw, 1, constant_values=True)
    return sliding_window_view(p, (3,3)).min(axis=(2,3))
def dilate(bw):
    p = np.pad(bw, 1, constant_values=False)
    return sliding_window_view(p, (3,3)).max(axis=(2,3))
def opening(bw): return dilate(erode(bw))

# 一个块 + 散点; 开运算去散点后连通域应只剩 1 个
bw = np.zeros((8, 8), dtype=bool); bw[2:6, 2:6] = True; bw[0, 7] = True
clean = opening(bw)
_, n_before = connected_components(bw, 8)
_, n_after = connected_components(clean, 8)
print(f'清理前 {n_before} 个连通域 -> 开运算后 {n_after} 个')
assert n_before == 2 and n_after == 1, '开运算应去掉散点, 实例数 2->1'

def soft_dice_loss(prob, gt, eps=1.0):
    '''prob∈[0,1] 概率图; gt 二值。软 Dice loss = 1 - 2*Σpg/(Σp+Σg+eps)。'''
    p = prob.ravel(); g = gt.ravel().astype(float)
    inter = (p * g).sum()
    return 1 - (2*inter + eps) / (p.sum() + g.sum() + eps)

gt = np.zeros((8,8)); gt[2:6,2:6] = 1
perfect = gt.copy()
assert soft_dice_loss(perfect, gt) < 0.05, '完美预测 Dice loss≈0'
assert soft_dice_loss(np.zeros((8,8)), gt) > soft_dice_loss(perfect, gt), '全空预测 loss 更大'
# 空 gt + 空 pred: eps 保证不除零、loss≈0
assert soft_dice_loss(np.zeros((4,4)), np.zeros((4,4))) < 0.05, '都空时 loss≈0(eps 防除零)'
print('✅ 形态学清理 + 软 Dice loss 正确(完美≈0、空对空不崩)')

---
## ✏️ 练习 1：Dice 系数

实现 `my_dice(pred, gt)`（二值掩码）：`2*交 / (|pred|+|gt|)`，都空时返回 1.0。

In [ ]:
def my_dice(pred, gt):
    # TODO: pred,gt 转 bool; inter=(pred&gt).sum(); denom=pred.sum()+gt.sum();
    #   return 2*inter/denom if denom>0 else 1.0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
g = np.zeros((6,6),bool); g[1:4,1:4]=True
p = np.zeros((6,6),bool); p[2:5,2:5]=True
d = my_dice(p, g)
i = iou_mask(p, g)
assert abs(d - 2*i/(1+i)) < 1e-9, 'Dice 应满足与 IoU 的恒等式'
assert my_dice(g, g) == 1.0, '自身 Dice=1'
assert my_dice(np.zeros((3,3),bool), np.zeros((3,3),bool)) == 1.0, '都空=1'
print('✅ 练习 1 通过：Dice 正确')

## ✏️ 练习 2：mIoU（跳过空类）

实现 `my_miou(pred, gt, K)`：对每类算 IoU，**该类在 pred、gt 都不出现就跳过**（不计入平均），返回有效类的平均。

In [ ]:
def my_miou(pred, gt, K):
    # TODO: 对 c in range(K): p=(pred==c); g=(gt==c); union=(p|g).sum()
    #   若 union>0: 把 (p&g).sum()/union 收集; 最后返回收集值的平均(空则 1.0)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
gt = np.array([[0,0,1],[0,1,1]])
pred = np.array([[0,0,1],[0,0,1]])
m = my_miou(pred, gt, 3)        # 类2 空, 应跳过
assert not np.isnan(m), '不应 NaN(空类跳过)'
assert 0 <= m <= 1
# 完美预测 mIoU=1
assert abs(my_miou(gt, gt, 3) - 1.0) < 1e-9, '完美预测 mIoU=1'
print(f'✅ 练习 2 通过：mIoU={m:.3f}, 空类已跳过')

## ✏️ 练习 3：连通域计数

实现 `count_components(bw, connectivity)`：返回二值图中连通域的**个数**（4 或 8 邻接）。可复用泛洪思路。

In [ ]:
def count_components(bw, connectivity=4):
    # TODO: 用 connected_components(bw, connectivity) 取第二个返回值, 或自己写泛洪计数
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
bw = np.zeros((5,5),bool); bw[0,0]=True; bw[0,2]=True; bw[4,4]=True  # 3 个孤立点
assert count_components(bw, 4) == 3, '三个孤立点 = 3 个连通域'
line = np.zeros((5,5),bool); line[2,:]=True                          # 一整行相连
assert count_components(line, 4) == 1, '一整行 = 1 个连通域'
print('✅ 练习 3 通过：连通域计数正确')

## ✏️ 练习 4：固定阈值分割

实现 `threshold_segment(img, t)`：返回 `img > t` 的二值前景掩码。是分割最朴素的一步。

In [ ]:
def threshold_segment(img, t):
    # TODO: return img > t
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
img = np.array([[0.1, 0.6],[0.9, 0.2]])
m = threshold_segment(img, 0.5)
assert m.dtype == bool and m.shape == img.shape
assert m[0,1] and m[1,0] and not m[0,0] and not m[1,1]
assert threshold_segment(img, 0.5).sum() == 2
print('✅ 练习 4 通过：阈值分割正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_dice(pred, gt):
    pred = pred.astype(bool); gt = gt.astype(bool)
    inter = (pred & gt).sum(); denom = pred.sum() + gt.sum()
    return 2*inter/denom if denom > 0 else 1.0

In [ ]:
# 练习 2 参考答案
def my_miou(pred, gt, K):
    vals = []
    for c_ in range(K):
        p = (pred == c_); g = (gt == c_); union = (p | g).sum()
        if union > 0:
            vals.append((p & g).sum() / union)
    return float(np.mean(vals)) if vals else 1.0

In [ ]:
# 练习 3 参考答案
def count_components(bw, connectivity=4):
    return connected_components(bw, connectivity)[1]

In [ ]:
# 练习 4 参考答案
def threshold_segment(img, t):
    return img > t

---
## 🧪 真实数据胶囊：optdigits 分割全管线

把真实 optdigits 数字图当作「前景 vs 背景」分割任务：Otsu 阈值 → 形态学清理 → 连通域 → 对真值掩码算 Dice/mIoU。

In [ ]:
def load_digits_or_synth(n=20, seed=0):
    try:
        from sklearn.datasets import load_digits
        d = load_digits(); return d.images[:n].astype(float)/16.0, 'real optdigits'
    except Exception:
        r = np.random.default_rng(seed)
        X = np.zeros((n,8,8))
        for i in range(n):
            cy,cx = r.integers(2,6,2); X[i, cy-1:cy+2, cx-1:cx+2] = 1.0
        return X, 'synthetic fallback'

digs, src = load_digits_or_synth(20)
print('source:', src)
# 真值前景: 像素 > 0.3(数字笔画); 预测: Otsu 阈值。
# 注: 8x8 数字笔画很细(1~2px), 开运算会侵蚀掉, 故这里直接用 Otsu 阈值不做开运算。
dices, mious = [], []
for im in digs[:10]:
    gt_fg = (im > 0.3).astype(int)
    if gt_fg.sum() == 0:
        continue
    t = otsu_threshold(im)
    pred_fg = (im > t).astype(int)
    dices.append(dice_mask(pred_fg, gt_fg))
    mious.append(mean_iou(pred_fg, gt_fg, 2))   # 2 类: 背景/前景
print(f'平均 Dice={np.mean(dices):.3f}  平均 mIoU={np.mean(mious):.3f}  (n={len(dices)} 张)')
assert len(dices) > 0
assert 0 <= np.mean(dices) <= 1 and 0 <= np.mean(mious) <= 1
assert np.mean(dices) > 0.4, 'Otsu 分割数字应有合理重合度'
print('✅ 真实 optdigits 分割全管线(Otsu→指标)跑通')

**🧪 胶囊练习**：实现 `segment_dice(img, gt_fg_thr=0.3)`：对一张图用 Otsu 阈值得到前景，与 `img>gt_fg_thr` 的真值算 Dice，返回 Dice。

In [ ]:
def segment_dice(img, gt_fg_thr=0.3):
    # TODO: gt=(img>gt_fg_thr); t=otsu_threshold(img); pred=(img>t); return dice_mask(pred, gt)
    raise NotImplementedError

In [ ]:
# 自测
vals = [segment_dice(im) for im in digs[:5] if (im>0.3).sum()>0]
assert all(0 <= v <= 1 for v in vals), 'Dice 应在 [0,1]'
print(f'✅ 胶囊练习通过：前 5 张平均 Dice = {np.mean(vals):.3f}')

In [ ]:
# 📖 胶囊参考答案
def segment_dice(img, gt_fg_thr=0.3):
    gt = (img > gt_fg_thr).astype(int)
    t = otsu_threshold(img)
    pred = (img > t).astype(int)
    return dice_mask(pred, gt)

### 小结
- **IoU/Dice**：同 TP/FP/FN，Dice=2·IoU/(1+IoU)；Dice 对小前景更友好。
- **mIoU**：对**类别**(macro)平均；**空类必须跳过/置 1**，否则 0/0=NaN 污染全局。
- **Otsu**：选类间方差最大的阈值，最简单的自动分割。
- **连通域**：泛洪标记把语义掩码切成实例；4 vs 8 邻接影响实例数。
- **类不平衡**：软 Dice loss(用概率, 加 ε)对背景占多数鲁棒；连通域前先开运算去散点。

下一站：**模块 05 · 自监督视觉** —— 跳出有标签世界，无监督地学表示(SimCLR/MAE)。